This notebook completes the precision/recall analysis

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import altair as alt
from datetime import datetime

In [ ]:
sge_ppj_subset = pd.read_excel('../Data/filtered_ppj_data/20251202_SGEsubset.xlsx')
vampseq_ppj_subset = pd.read_excel('../Data/filtered_ppj_data/20251204_VAMPseqsubset_wDups.xlsx')

dfs = {'SGE': sge_ppj_subset,
       'VAMP-seq': vampseq_ppj_subset
      }

alt.data_transformers.disable_max_rows() #gets rid of max data length problem

In [ ]:
def process_dfs(dfs): #Reads and processes dataframes

    keys = list(dfs.keys())
    cleaned_dfs = {} #dictionary to store cleaned dataframes
    for key in keys:
        df = dfs[key]
        df = df.rename(columns = {'auth_reported_func_class': 'functional_consequence',
                                   'clinvar_sig': 'Germline classification'}
                      )

        df = df.dropna(subset = ['functional_consequence'])
        df = df.dropna(subset = ['Germline classification'])

        df = df.loc[(df['functional_consequence'] != 'indeterminate') & (~df['Germline classification'].isin(['Uncertain significance', 'Conflicting classifications of pathogenicity']))]

        cleaned_dfs[key] = df

    return cleaned_dfs

In [ ]:
def wilson_interval(successes, total, confidence=0.95): #Custom function for calculating 95% CI
    """
    Calculate Wilson score confidence interval for a proportion
    """
    if total == 0:
        return 0, 0, 0
    
    p = successes / total
    z = stats.norm.ppf((1 + confidence) / 2)
    
    denominator = 1 + z**2 / total
    center = (p + z**2 / (2 * total)) / denominator
    margin = z * np.sqrt((p * (1 - p) / total + z**2 / (4 * total**2))) / denominator
    
    lower = center - margin
    upper = center + margin
    
    return p, lower, upper

In [ ]:
def clinvar_performance(df, assay_name):
    """Calculate precision/recall metrics and return them (no plotting)."""
    
    df = df.loc[~df['functional_consequence'].isin(['indeterminate'])]
    df = df.loc[~df['Germline classification'].isin(['Uncertain significance', 'Conflicting classifications of pathogenicity', 'no classification for the single variant', 'not provided'])]
    df = df.dropna(subset=['Germline classification'])
    df['test'] = 0

    # Group BLB and PLP variants
    df.loc[(df['Germline classification'] == 'Benign') | (df['Germline classification'] == 'Likely benign') | (df['Germline classification'] == 'Benign/Likely benign') | (df['Germline classification'] == 'Benign; association'), 'simple_classification'] = 'Benign'
    df.loc[(df['Germline classification'] == 'Pathogenic') | (df['Germline classification'] == 'Likely pathogenic') | (df['Germline classification'] == 'Pathogenic/Likely pathogenic'), 'simple_classification'] = 'Pathogenic'

    # True benign/pathogenic dataframes
    benign_df = df.loc[df['simple_classification'].isin(['Benign'])]
    path_df = df.loc[df['simple_classification'].isin(['Pathogenic'])]

    # Assay benign/pathogenic dataframes
    assay_benign_df = df.loc[df['functional_consequence'].isin(['functionally_normal'])].copy()
    assay_path_df = df.loc[df['functional_consequence'].isin(['functionally_abnormal'])].copy()

    # Agreement tests
    benign_df.loc[benign_df['functional_consequence'] == 'functionally_normal', 'test'] = 1
    path_df.loc[path_df['functional_consequence'] == 'functionally_abnormal', 'test'] = 1
    assay_benign_df.loc[assay_benign_df['simple_classification'] == 'Benign', 'test'] = 1 
    assay_path_df.loc[assay_path_df['simple_classification'] == 'Pathogenic', 'test'] = 1

    # Calculate metrics with confidence intervals
    npv_successes = assay_benign_df['test'].sum()
    npv_total = len(assay_benign_df)
    raw_npv, npv_lower, npv_upper = wilson_interval(npv_successes, npv_total)
    
    ppv_successes = assay_path_df['test'].sum()
    ppv_total = len(assay_path_df)
    raw_ppv, ppv_lower, ppv_upper = wilson_interval(ppv_successes, ppv_total)

    recall_successes = path_df['test'].sum()
    recall_total = len(path_df)
    recall, recall_lower, recall_upper = wilson_interval(recall_successes, recall_total)

    print(f'{assay_name}:')
    print(f'  Precision/PPV: {raw_ppv:.3f} (95% CI: {ppv_lower:.3f}-{ppv_upper:.3f})')
    print(f'  Recall/Sensitivity: {recall:.3f} (95% CI: {recall_lower:.3f}-{recall_upper:.3f})')
    print(f'  NPV: {raw_npv:.3f} (95% CI: {npv_lower:.3f}-{npv_upper:.3f})\n')
    
    return {
        'assay': assay_name,
        'precision': raw_ppv,
        'precision_lower': ppv_lower,
        'precision_upper': ppv_upper,
        'recall': recall,
        'recall_lower': recall_lower,
        'recall_upper': recall_upper,
        'npv': raw_npv,
        'npv_lower': npv_lower,
        'npv_upper': npv_upper
    }


def plot_precision_recall_panels(metrics_list):
    """Create faceted bar plots for precision/recall using Altair."""
    
    # Build dataframe from metrics
    rows = []
    for m in metrics_list:
        rows.append({
            'assay': m['assay'],
            'metric': 'Precision',
            'value': m['precision'],
            'lower_ci': m['precision_lower'],
            'upper_ci': m['precision_upper']
        })
        rows.append({
            'assay': m['assay'],
            'metric': 'Recall',
            'value': m['recall'],
            'lower_ci': m['recall_lower'],
            'upper_ci': m['recall_upper']
        })
    
    plot_df = pd.DataFrame(rows)
    
    # Add formatted label
    plot_df['label'] = plot_df.apply(
        lambda row: f"{row['value']:.2f} ({row['lower_ci']:.2f}-{row['upper_ci']:.2f})", 
        axis=1
    )
    
    # Add midpoint for text positioning
    plot_df['text_y'] = plot_df['value'] / 2
    
    # Get assay order from input
    assay_order = [m['assay'] for m in metrics_list]
    
    # Bars
    bars = alt.Chart(plot_df).mark_bar(color='#cfcfcf').encode(
        x=alt.X('metric:N',
                title='',
                sort=['Precision', 'Recall'],
                axis=alt.Axis(labelFontSize=14, labelFontWeight='bold')),
        y=alt.Y('value:Q',
                title='',
                scale=alt.Scale(domain=[0, 1]),
                axis=alt.Axis(labels=True, ticks=True, tickSize=3, title = '', labelFontSize = 16))
    )
    
    # Error bars
    error_bars = alt.Chart(plot_df).mark_errorbar(thickness=2).encode(
        x=alt.X('metric:N', sort=['Precision', 'Recall']),
        y=alt.Y('lower_ci:Q'),
        y2=alt.Y2('upper_ci:Q'),
        color=alt.value('black')
    )
    
    '''
    # Text labels inside bars, vertical
    text = alt.Chart(plot_df).mark_text(
        align='center',
        baseline='middle',
        angle=270,
        fontSize=16,
        fontWeight='bold',
        color='black'
    ).encode(
        x=alt.X('metric:N', sort=['Precision', 'Recall']),
        y=alt.Y('text_y:Q', scale=alt.Scale(domain=[0, 1])),
        text='label:N'
    )

    '''
    # Combine and facet by assay
    plot = (bars + error_bars).properties(
        width=120,
        height=325
    ).facet(
        column=alt.Column('assay:N',
                          title=None,
                          header=alt.Header(labelFontSize=16, labelFontWeight='bold'),
                          sort=assay_order)
    ).configure_axis(
        grid=False
    ).configure_view(
        stroke=None
    )
    
    plot.display()
    
    return plot


In [ ]:
def main(save = False):
    processed_dfs = process_dfs(dfs)

    keys = ['VAMP-seq', 'SGE']
    
    # Usage:
    #keys = list(processed_dfs.keys())
    metrics_list = []
    
    for key in keys:
        df = processed_dfs[key]
        metrics = clinvar_performance(df, key)
        metrics_list.append(metrics)
    
    final_plot = plot_precision_recall_panels(metrics_list)

    if save: 
        date = datetime.today().strftime('%Y%m%d')
        save_string = f'/Users/ivan/Desktop/pillar_project_figs/{date}_PillarProject_PRvsClinVar_wErrorBar_grey.svg'

        final_plot.save(save_string)

In [ ]:
main(save = False)